Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/training-with-deep-learning/train-hyperparameter-tune-deploy-with-tensorflow/train-hyperparameter-tune-with-notebook.png)

# Training, hyperparameter tune a notebook with TensorFlow

## Introduction
This tutorial shows how to submit a jupyter notebook .ipynb file (instead of python .py file) to train a simple deep neural network using the MNIST dataset and TensorFlow on Azure Machine Learning. MNIST is a popular dataset consisting of 70,000 grayscale images. Each image is a handwritten digit of `28x28` pixels, representing number from 0 to 9. The goal is to create a multi-class classifier to identify the digit each image represents, and deploy it as a web service in Azure.
This is a template that can run any notebook with tweaks on parameters.

For more information about the MNIST dataset, please visit [Yan LeCun's website](http://yann.lecun.com/exdb/mnist/).

## Prerequisite:
* Understand the [architecture and terms](https://docs.microsoft.com/azure/machine-learning/service/concept-azure-machine-learning-architecture) introduced by Azure Machine Learning
* Go through the [configuration notebook](../../../configuration.ipynb) to:
    * install the AML SDK
    * create a workspace and its configuration file (`config.json`)
    * create a gpu cluster compute target

In [ ]:
import azureml.core
import os
import shutil
import sys
import urllib

from azureml.core import Workspace, Experiment
from azureml.contrib.notebook.notebook_run_config import NotebookRunConfig
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.runconfig import RunConfiguration, DEFAULT_GPU_IMAGE
from azureml.train.dnn import TensorFlow
from azureml.train.hyperdrive import choice, loguniform
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.runconfig import HyperDriveConfig, PrimaryMetricGoal
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.widgets import RunDetails
from os import path, makedirs

sys.path.append("../")

# check core SDK version number
print("Azure ML SDK Version: ", azureml.core.VERSION)

### Initialize workspace
Initialize a [Workspace](https://docs.microsoft.com/azure/machine-learning/service/concept-azure-machine-learning-architecture#workspace) object from the existing workspace you created in the Prerequisites step. `Workspace.setup()` reads `config.json` if it exist or tries to load existing workspace from your resource group with interactive prompts.

In [ ]:
ws = Workspace.setup()

### Create an Azure ML experiment
Let's create an experiment named "tf-mnist" and a folder to hold the training scripts. The script runs will be recorded under the experiment in Azure.

In [ ]:
exp = Experiment(workspace=ws, name='tf-mnist-notebook')

### Download MNIST dataset
In order to train on the MNIST dataset we will first need to download it from Yan LeCun's web site directly and save them in a `data` folder locally.

In [ ]:
local_data_dir = './data/mnist'
os.makedirs(local_data_dir, exist_ok=True)

urllib.request.urlretrieve('http://yann.lecun.com/exdb/mnist/train-images-idx3-ubyte.gz', filename = './data/mnist/train-images.gz')
urllib.request.urlretrieve('http://yann.lecun.com/exdb/mnist/train-labels-idx1-ubyte.gz', filename = './data/mnist/train-labels.gz')
urllib.request.urlretrieve('http://yann.lecun.com/exdb/mnist/t10k-images-idx3-ubyte.gz', filename = './data/mnist/test-images.gz')
urllib.request.urlretrieve('http://yann.lecun.com/exdb/mnist/t10k-labels-idx1-ubyte.gz', filename = './data/mnist/test-labels.gz')

### Upload MNIST dataset to default datastore 
A [datastore](https://docs.microsoft.com/azure/machine-learning/service/how-to-access-data) is a place where data can be stored that is then made accessible to a Run either by means of mounting or copying the data to the compute target. A datastore can either be backed by an Azure Blob Storage or an Azure File Share (ADLS will be supported in the future). For simple data handling, each workspace provides a default datastore that can be used, in case the data is not already in Blob Storage or File Share.

In [ ]:
ds = ws.get_default_datastore()

In this next step, we will upload the training and test set into the workspace's default datastore, which we will later mount on an AmlCompute cluster for training.

In [ ]:
ds.upload(src_dir=local_data_dir, target_path='mnist', overwrite=True, show_progress=True)

# Train with TensorFlow Estimator
Next, we construct an `azureml.train.dnn.TensorFlow` estimator object, use the previously created gpu cluster as compute target, and pass the mount-point of the datastore to the training code as a parameter.

The TensorFlow estimator provides a simple way of launching a TensorFlow training job on a compute target. It will automatically provide a docker image that has TensorFlow installed. Below is a table of popular parameters or see the [SDK documentation](https://docs.microsoft.com/en-us/python/api/azureml-train-core/azureml.train.dnn.tensorflow?view=azure-ml-py).

| Parameter  | Description |
| ------------- | ------------- |
| source_directory  | (str) Local directory that contains all of your code needed for the training job. This folder gets copied from your local machine to the remote compute  |
| script_params | (dict) A dictionary containing parameters to the `entry_script`. This is useful for passing parameters including datastore reference, for example, see [train-hyperparameter-tune-deploy-with-tensorflow.ipynb](../train-hyperparameter-tune-deploy-with-tensorflow/train-hyperparameter-tune-deploy-with-tensorflow.ipynb) |
| compute_target  | ([AbstractComputeTarget](https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.compute_target.abstractcomputetarget?view=azure-ml-py) or str) Remote compute target that your training script will run on, in this case a previously created persistent compute cluster (cpu_cluster) |
| entry_script  | (str) Filepath (relative to the source_directory) of the training script/notebook to be run on the remote compute. This file, and any additional files it depends on, should be located in this folder  |
| use_gpu | (bool) A bool value indicating if the environment to run the experiment should support GPUs. If set to true, gpu-based default docker image will be used in the environment. If set to false, CPU based image will be used. Default docker images (CPU or GPU) will be used only if custom_docker_image parameter is not set. This setting is used only in docker enabled compute targets|
| pip_packages | (list) List of strings representing pip packages to be added to the Python environment for the experiment|
| conda_packages | (list) List of strings representing conda packages to be added to the Python environment for the experiment|
| framework_version | (str) Tensorflow version to be used for executing training code. TensorFlow.get_supported_versions() returns a list of the versions supported by the current SDK. If no version is provided, the estimator will default to the latest version supported by AzureML|

In [ ]:
script_params = {
    'data_folder': ds.as_mount(),
    'batch_size': 50,
    'n_hidden_1': 300,
    'n_hidden_2': 100
}
tf_est = TensorFlow(source_directory='.',
                 script_params=script_params,
                 compute_target=gpu_cluster,
                 entry_script='tf_mnist.ipynb', 
                 use_gpu=True,
                 pip_packages=['tensorflow']
                )

### Submit estimator and monitor the job run
Use `submit()` to submit estimator to an AzureML experiment. `RunDetails` widget shows status of current run in jupyter notebook. See more details about monitoring run in [**Monitor the Run**](../train-hyperparameter-tune-deploy-with-tensorflow/train-hyperparameter-tune-deploy-with-tensorflow.ipynb#monitor-run) and [**The Run object**](../train-hyperparameter-tune-deploy-with-tensorflow/train-hyperparameter-tune-deploy-with-tensorflow.ipynb#run-object) section from [Training, hyperparameter tune, and deploy with TensorFlow](../train-hyperparameter-tune-deploy-with-tensorflow/train-hyperparameter-tune-deploy-with-tensorflow.ipynb) notebook or [Manage runs](../../training/manage-runs/manage-runs.ipynb) notebook.

In [ ]:
run = exp.submit(tf_est)
RunDetails(run).show()

# Intelligent hyperparameter tuning
We have trained the model with one set of hyperparameters, now let's see how we can do hyperparameter tuning by launching multiple runs on the cluster. First let's define the parameter space using random sampling. They will overwrite parameters with the same name from `script_params`.
Here're some popular parameters or please see the [SDK documentation](https://docs.microsoft.com/en-us/python/api/azureml-train-core/azureml.train.hyperdrive.hyperdriveconfig?view=azure-ml-py).

| Parameter  | Description |
| ------------- | ---- |
| estimator  | ([MMLBaseEstimator](https://docs.microsoft.com/en-us/python/api/azureml-train-core/azureml.train.estimator.mmlbaseestimator?view=azure-ml-py)) An estimator that will be called with sampled hyper parameters |
| hyperparameter_sampling | ([HyperParameterSampling](https://docs.microsoft.com/en-us/python/api/azureml-train-core/azureml.train.hyperdrive.hyperparametersampling?view=azure-ml-py)) The hyperparameter sampling space |
| policy  | ([EarlyTerminationPolicy](https://docs.microsoft.com/en-us/python/api/azureml-train-core/azureml.train.hyperdrive.earlyterminationpolicy?view=azure-ml-py)) The early termination policy to use. If None - the default, no early termination policy will be used. The MedianTerminationPolicy with delay_evaluation of 5 is a good termination policy to start with. These are conservative settings, that can provide 25%-35% savings with no loss on primary metric (based on our evaluation data) |
| primary_metric_name  | (str) The name of the primary metric reported by the experiment runs |
| primary_metric_goal | ([PrimaryMetricGoal](https://docs.microsoft.com/en-us/python/api/azureml-train-core/azureml.train.hyperdrive.primarymetricgoal?view=azure-ml-py)) One of maximize / minimize. It determines if the primary metric has to be minimized/maximized in the experiment runs' evaluation |
| max_total_runs | (int) Maximum number of runs. This is the upper bound; there may be fewer runs when the sample space is smaller than this value |
| max_concurrent_runs | (int) Maximum number of runs to run concurrently. If None, all runs are launched in parallel |

In [ ]:
ps = RandomParameterSampling(
    {
        '--batch_size': choice(25, 50, 100),
        '--n_hidden_1': choice(10, 50, 200, 300, 500),
        '--n_hidden_2': choice(10, 50, 200, 500),
        '--learning_rate': loguniform(-6, -1)
    }
)
early_termination_policy = BanditPolicy(slack_factor = 0.15, evaluation_interval=10)
htc = HyperDriveConfig(estimator=tf_est, 
                       hyperparameter_sampling=ps, 
                       policy=early_termination_policy, 
                       primary_metric_name='validation_acc', 
                       primary_metric_goal=PrimaryMetricGoal.MAXIMIZE, 
                       max_total_runs=8,
                       max_concurrent_runs=4)


Finally, let's launch the hyperparameter tuning job. We can use a run history widget to show the progress. Be patient as this might take more than 10 mins to complete.

In [ ]:
hd_run = exp.submit(htc)
RunDetails(hd_run).show()

## Register and deploy model
To find your best model, register and deploy, you can follow the steps described in [**Find and register best model**](../train-hyperparameter-tune-deploy-with-tensorflow/train-hyperparameter-tune-deploy-with-tensorflow.ipynb#register-model) section in [Training, hyperparameter tune, and deploy with TensorFlow](../train-hyperparameter-tune-deploy-with-tensorflow/train-hyperparameter-tune-deploy-with-tensorflow.ipynb) notebook.